# AutoCitation — Extractor Fine-tune (phi3:mini, QLoRA via Unsloth)

**Before running:** Runtime → Change runtime type → **T4 GPU**.

Expects `train.jsonl` / `val.jsonl` (from `build_dataset.py`) in a Google Drive folder named `AutoCitation_finetune`.

Output: a `.gguf` file copied back to the same Drive folder → load into Ollama locally.

In [ ]:
# 1. Install (takes ~2 min on Colab)
%pip install -q unsloth


In [ ]:
# 2. Mount Google Drive and locate the dataset
from google.colab import drive
drive.mount('/content/drive')

DATA_DIR   = '/content/drive/MyDrive/AutoCitation_finetune'
TRAIN_PATH = f'{DATA_DIR}/train.jsonl'
VAL_PATH   = f'{DATA_DIR}/val.jsonl'

import os
assert os.path.exists(TRAIN_PATH), f'train.jsonl not found at {TRAIN_PATH} — upload it to Drive first.'
print('Dataset found.')


In [ ]:
# 3. Load base model in 4-bit (QLoRA)
from unsloth import FastLanguageModel

MAX_SEQ_LEN = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = 'unsloth/Phi-3-mini-4k-instruct',
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    target_modules = ['q_proj', 'k_proj', 'v_proj', 'o_proj',
                      'gate_proj', 'up_proj', 'down_proj'],
    use_gradient_checkpointing = 'unsloth',
    random_state = 42,
)


In [ ]:
# 4. Load dataset and format as chat (user = production prompt, assistant = Fact_N / Terminate)
from datasets import load_dataset

ds = load_dataset('json', data_files={'train': TRAIN_PATH, 'val': VAL_PATH})

def to_chat(example):
    messages = [
        {'role': 'user',      'content': example['prompt']},
        {'role': 'assistant', 'content': example['completion']},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

ds = ds.map(to_chat)
print(ds)
print(ds['train'][0]['text'][:600])


In [ ]:
# 5. Train (~1-2h for a few thousand examples on T4)
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = ds['train'],
    eval_dataset       = ds['val'],
    dataset_text_field = 'text',
    max_seq_length     = MAX_SEQ_LEN,
    args = TrainingArguments(
        output_dir                  = 'outputs',
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        num_train_epochs            = 2,
        learning_rate               = 2e-4,
        lr_scheduler_type           = 'cosine',
        warmup_ratio                = 0.05,
        logging_steps               = 10,
        eval_strategy               = 'steps',
        eval_steps                  = 100,
        save_strategy               = 'no',
        fp16                        = True,
        seed                        = 42,
    ),
)

trainer.train()


In [ ]:
# 6. Sanity test: faithfulness + format on an unseen sentence
FastLanguageModel.for_inference(model)

test_prompt = (
    'You are an expert Atomic Fact Extractor. Your task is to extract the VERY FIRST '
    'single, verifiable factual unit from the text.\n\n'
    'Rules:\n'
    '- Isolate one checkable fact expressing a single subject-relation-object relationship.\n'
    '- Write the fact as ONE plain declarative sentence.\n'
    '- Use ONLY information stated in the text.\n'
    '- Output exactly ONE line in this format and nothing else: Fact_1: <extracted fact>\n\n'
    'Text:\n\"\"\"The Bosphorus Bridge, which opened in 1973, connects the European and Asian sides of Istanbul.\"\"\"\n\n'
    'Extract Fact_1:'
)

inputs  = tokenizer.apply_chat_template(
    [{'role': 'user', 'content': test_prompt}],
    return_tensors='pt', add_generation_prompt=True,
).to('cuda')
outputs = model.generate(inputs, max_new_tokens=64, temperature=0.1)
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))


In [ ]:
# 7. Export merged GGUF (q4_k_m ≈ 2.3 GB) and copy to Drive
model.save_pretrained_gguf('gguf_out', tokenizer, quantization_method='q4_k_m')

# Unsloth writes the merged 16-bit HF weights into the folder you pass
# ('gguf_out') but writes the FINAL quantized .gguf into a SEPARATE sibling
# folder named '<folder>_gguf' (see the "Generated files: [...]" line in
# the conversion log above) — so searching only 'gguf_out/*.gguf' can come
# up empty depending on the installed Unsloth version. Check both known
# locations, then fall back to a recursive search, and fail with a clear
# message instead of a bare IndexError if nothing is found anywhere.
import glob, os, shutil

candidates = []
for pattern in ['gguf_out_gguf/*.gguf', 'gguf_out/*.gguf', '**/*.gguf']:
    candidates = glob.glob(pattern, recursive=True)
    if candidates:
        break

if not candidates:
    raise FileNotFoundError(
        'No .gguf file found under gguf_out_gguf/, gguf_out/, or anywhere '
        'else. Scroll up to the conversion log and look for a line like '
        '"Generated files: [...]" to find the real path, then set '
        "gguf = '<that path>' manually and rerun from the next line."
    )

gguf = candidates[0]
print(f'Found: {gguf} ({os.path.getsize(gguf) / 1e9:.2f} GB)')

dest = f'{DATA_DIR}/autocitation-extractor.q4_k_m.gguf'
shutil.copy(gguf, dest)
print('Saved to Drive:', dest)
print('Next: download it and follow README step 5 (Ollama Modelfile).')


In [ ]:
# 8. Free disk space — run AFTER confirming the .gguf above is on Drive.
#
# The merge + GGUF conversion leaves several large intermediates on the
# Colab VM's local disk you no longer need once the final q4_k_m file is
# saved to Drive: the merged 16-bit weights ('gguf_out/'), the f16
# intermediate GGUF and llama.cpp build ('gguf_out_gguf/'), any trainer
# checkpoints ('outputs/'), and the HuggingFace hub cache (the base model
# shards that got re-downloaded during export — visible in the log above
# as 'Cache check failed... will proceed with downloading'). Together
# these can easily total 20-30 GB on a filesystem that may only have
# ~35-78 GB to begin with.
print('Before cleanup:')
!df -h /content

import shutil, os

for path in ['gguf_out', 'gguf_out_gguf', 'outputs']:
    if os.path.exists(path):
        shutil.rmtree(path, ignore_errors=True)
        print(f'Removed ./{path}')

!rm -rf /root/.cache/huggingface/hub
!pip cache purge

print('\nAfter cleanup:')
!df -h /content
